# TAMER - Tree-Aware Transformer for Math OCR
## Google Colab Training Script with Curriculum Learning

This notebook implements **Curriculum Learning by Domain** - training the model
progressively from easy to hard data, exactly like teaching a human.

### Training Strategy: 3-Stage Curriculum

| Stage | Dataset | Source | Type | Purpose |
|-------|---------|--------|------|--------|
| 1 | im2latex-100k | Kaggle | Perfect printed text | Learn LaTeX grammar and symbols |
| 2 | mathwriting | Hugging Face | Clean handwriting | Bridge to human penmanship |
| 3 | crohme + hme100k | Zenodo/GitHub | Messy handwriting | Robust real-world training |

### Prerequisites
- GPU runtime (Runtime -> Change runtime type -> GPU -> T4 or A100)
- Kaggle API token
- Hugging Face token (for some datasets)

### Step 1: Clone Repository and Install Dependencies

In [ ]:
# ============================================================
# Step 1: Clone/update repo and install dependencies
# ============================================================
import os

repo_path = '/content/AI_PROJECT_TAMER_Complete'

if os.path.exists(repo_path):
    # Fresh pull of latest main (matches GitHub exactly)
    !cd /content/AI_PROJECT_TAMER_Complete && git fetch origin && git reset --hard origin/main
    print('Reset to origin/main.')
else:
    %cd /content
    !git clone --depth 1 https://github.com/Vtheonly/AI_PROJECT_TAMER_Complete.git
    print('Cloned repository.')

%cd /content/AI_PROJECT_TAMER_Complete/tamer_ocr
!pip install --quiet -r requirements.txt

!git log -1 --format='Latest commit: %h - %s (%cr)'
!echo '---'
!ls -la

### Step 2: Configure Authentication

You need authentication tokens to download datasets:

**Kaggle Token** (for im2latex-100k):
1. Go to https://www.kaggle.com/settings/account
2. Click Create New API Token

**Hugging Face Token** (for mathwriting):
1. Go to https://huggingface.co/settings/tokens
2. Create a new token

In [ ]:
# ============================================================
# Step 2: Configure Authentication Tokens
# ============================================================
import os

# Try loading from Colab Secrets first (recommended - more secure)
# Go to Colab -> Keys icon -> Add new secret
try:
    from google.colab import userdata
    
    # Kaggle token
    kaggle_token = userdata.get('KAGGLE_API_TOKEN')
    if kaggle_token:
        os.environ['KAGGLE_API_TOKEN'] = kaggle_token
        print('Kaggle token loaded from Colab Secrets.')
    
    # Hugging Face token
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        print('Hugging Face token loaded from Colab Secrets.')
except Exception:
    pass

# Interactive input for missing tokens
from getpass import getpass

if not os.environ.get('KAGGLE_API_TOKEN'):
    token = getpass('Enter Kaggle API Token (starts with KGAT_): ')
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
        print('Kaggle token configured.')

if not os.environ.get('HF_TOKEN'):
    token = getpass('Enter Hugging Face Token (optional, for mathwriting dataset): ')
    if token:
        os.environ['HF_TOKEN'] = token
        print('Hugging Face token configured.')

# Verify
print('\nAuthentication Status:')
print('  Kaggle API Token:', 'SET' if os.environ.get('KAGGLE_API_TOKEN') else 'NOT SET')
print('  Hugging Face Token:', 'SET' if os.environ.get('HF_TOKEN') else 'NOT SET')

### Step 3 (Colab-Only): Prepare datasets first (Download-Only)

**Do NOT run this on your local PC/laptop.** This step can download and process very large datasets and may crash machines with limited RAM/disk.

Run this **only in Google Colab** (GPU runtime recommended). It will **download/parse/validate** the datasets and (if your HF token is configured) **push the verified dataset** to your Hugging Face dataset repo.

**Step A: Prepare-only (no training)**

```bash
!python train.py --datasets im2latex-100k mathwriting crohme hme100k --prepare-only
```

**Step B: Train (after prepare-only finishes)**

```bash
!python train.py --download --datasets im2latex-100k --num-epochs 50
```

### STAGE 1: Foundation Training (Perfect Printed Text)

**Dataset:** im2latex-100k  
**Source:** Kaggle  
**Purpose:** Learn LaTeX grammar, symbol recognition, and basic structure  
**Recommended:** 50 epochs

Run this cell first. The model will learn from perfect, computer-rendered
LaTeX formulas. This teaches the model the grammar of mathematical notation
without the confusion of messy handwriting.

In [ ]:
# ============================================================
# STAGE 1: Pre-train on Printed Text (im2latex-100k)
# ============================================================
# This will:
# 1. Download the im2latex-100k dataset from Kaggle (if not already downloaded)
# 2. Train the model from scratch
# 3. Save checkpoint to checkpoints/latest.pt
# ============================================================

!python train.py --download --datasets im2latex-100k --num-epochs 50

print('\nStage 1 Complete!')
print('Model has learned LaTeX grammar from perfect printed text.')
print('Checkpoint saved to: checkpoints/latest.pt')
print('\nNext: Run Stage 2 cell below to fine-tune on clean handwriting.')

### STAGE 2: Fine-tuning on Clean Handwriting

**Dataset:** mathwriting  
**Source:** Hugging Face  
**Purpose:** Bridge from printed text to human handwriting  
**Recommended:** 30 epochs

The model loads the checkpoint from Stage 1 and adapts to clean,
digital handwriting. It already knows LaTeX grammar - now it learns
that lines are not perfectly straight anymore.

In [ ]:
# ============================================================
# STAGE 2: Fine-tune on Clean Handwriting (mathwriting)
# ============================================================
# This will:
# 1. Download the mathwriting dataset from Hugging Face
# 2. Load checkpoint from Stage 1 (checkpoints/latest.pt)
# 3. Continue training on clean handwriting data
# ============================================================

!python train.py --download --datasets mathwriting --num-epochs 30

print('\nStage 2 Complete!')
print('Model can now handle clean handwriting.')
print('Checkpoint updated: checkpoints/latest.pt')
print('\nNext: Run Stage 3 cell below for final polishing on messy handwriting.')

### STAGE 3: Final Polish on Messy Handwriting

**Datasets:** crohme + hme100k  
**Sources:** Zenodo (CROHME) and GitHub (HME100K)  
**Purpose:** Make the model robust against real-world messy handwriting  
**Recommended:** 50 epochs

This is the hardest data. Because the model already knows LaTeX grammar
(Stage 1) and basic handwriting (Stage 2), it can focus on adapting to
messy, overlapping, varying-thickness real-world handwriting.

In [ ]:
# ============================================================
# STAGE 3: Final Polish on Messy Handwriting (crohme + hme100k)
# ============================================================
# This will:
# 1. Download CROHME from Zenodo and HME100K from GitHub
# 2. Load checkpoint from Stage 2
# 3. Final training on messy, real-world handwriting
# ============================================================

!python train.py --download --datasets crohme hme100k --num-epochs 50

print('\nStage 3 Complete! Training Finished.')
print('Model is now trained on all difficulty levels.')
print('Final checkpoint: checkpoints/latest.pt')

### Optional: Run Individual Stages

If you want to run stages independently or resume training, use these commands:

In [ ]:
# Run ONLY Stage 1 from scratch
# !python train.py --download --datasets im2latex-100k --num-epochs 50

# Resume training on a different dataset (loads latest checkpoint automatically)
# !python train.py --download --datasets mathwriting --num-epochs 30

# Train on multiple datasets at once
# !python train.py --download --datasets crohme hme100k --num-epochs 50

# Run without downloading (use if datasets already exist)
# !python train.py --datasets im2latex-100k --num-epochs 50

### Why This 3-Stage Approach Works

This is **Curriculum Learning by Domain** - the same strategy used by
state-of-the-art models like Nougat and TrOCR:

1. **Stage 1 teaches Syntax and Structure** - The model learns what LaTeX
   looks like, how brackets are balanced, and what symbols mean.

2. **Stage 2 transfers to Basic Human Strokes** - The model keeps its
   LaTeX knowledge and learns that human lines are not perfectly straight.

3. **Stage 3 hardens against Noise** - The model becomes robust to
   overlapping symbols, varying thickness, and messy writing.

If you tried to start at Stage 3, the model would fail because it would
have to learn LaTeX grammar AND bad handwriting simultaneously. By going
step-by-step, each stage builds on the previous one.

### Troubleshooting

**Kaggle authentication error:**
- Make sure your API token is correct (starts with KGAT_)
- Try re-generating the token on Kaggle settings
- Check token is set: `print(os.environ.get('KAGGLE_API_TOKEN'))`)

**Download fails for large files:**
- The downloader uses requests with retry logic - it handles network issues
- SSL errors: `!pip install --upgrade certifi`

**Git clone failed or git not found:**
- Colab has git pre-installed
- If not: `!apt-get install -y git`

**Out of memory errors:**
- Reduce batch_size in config.py
- Use A100 accelerator for more memory (Runtime -> Change runtime type)